#  NLP Assignment 3

## ID: i232607


In [ ]:
%pip install torch torchvision torchaudio

In [2]:
import os
import re
import json
import random
from collections import Counter

import numpy as np
import torch
from sklearn.model_selection import train_test_split

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Paths
DATA_DIR = "dataset"
MODELS_DIR = "models"
RESULTS_DIR = "results"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Folders ready:", MODELS_DIR, RESULTS_DIR)

In [ ]:
# Config for dataset and preprocessing
CATEGORY_FILES = {
    "beauty": "Beauty_5.json",
    "cellphones": "Cell_Phones_and_Accessories_5.json",
    "sports": "Sports_and_Outdoors_5.json",
}

SAMPLES_PER_CATEGORY = 10000   # change to 12000 or 15000 if needed
MAX_LEN = 120                  # fixed sequence length
MIN_FREQ = 2                   # min token frequency for vocab

print("Using", SAMPLES_PER_CATEGORY, "reviews per category")

In [ ]:
def read_reviews(path, category_name, limit):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
            except Exception:
                continue

            text = obj.get("reviewText", "")
            rating = obj.get("overall", None)

            if text is None:
                text = ""
            text = str(text).strip()

            if rating is None:
                continue
            try:
                rating = int(round(float(rating)))
            except Exception:
                continue

            if rating < 1 or rating > 5:
                continue
            if len(text) == 0:
                continue

            rows.append({
                "text": text,
                "rating": rating,
                "category": category_name
            })

    if len(rows) < limit:
        print(f"Warning: {category_name} has only {len(rows)} usable reviews")
        return rows

    random.shuffle(rows)
    return rows[:limit]


def rating_to_sentiment(r):
    if r <= 2:
        return 0  # negative
    if r == 3:
        return 1  # neutral
    return 2      # positive


all_rows = []
for cat, file_name in CATEGORY_FILES.items():
    full_path = os.path.join(DATA_DIR, file_name)
    part = read_reviews(full_path, cat, SAMPLES_PER_CATEGORY)
    all_rows.extend(part)
    print(cat, "->", len(part))

print("Total rows:", len(all_rows))

In [ ]:
# Train/val/test split: 70/15/15
train_rows, temp_rows = train_test_split(all_rows, test_size=0.30, random_state=SEED, shuffle=True)
val_rows, test_rows = train_test_split(temp_rows, test_size=0.50, random_state=SEED, shuffle=True)

print("Train:", len(train_rows))
print("Val:  ", len(val_rows))
print("Test: ", len(test_rows))


def show_label_stats(rows, name):
    sent_counts = Counter()
    cat_counts = Counter()
    for x in rows:
        sent_counts[rating_to_sentiment(x["rating"])] += 1
        cat_counts[x["category"]] += 1

    print(f"\n{name} sentiment counts (0=neg,1=neu,2=pos):", dict(sent_counts))
    print(f"{name} category counts:", dict(cat_counts))


show_label_stats(train_rows, "Train")
show_label_stats(val_rows, "Val")
show_label_stats(test_rows, "Test")

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s\.,!?;:'\"()\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    # Split words and punctuation separately
    tokens = re.findall(r"[a-z0-9]+|[^\w\s]", text)
    return tokens


def build_vocab(rows, min_freq=2):
    counter = Counter()
    for row in rows:
        txt = clean_text(row["text"])
        tokens = tokenize(txt)
        for t in tokens:
            counter[t] += 1

    vocab = {
        "<PAD>": 0,
        "<UNK>": 1
    }

    for token, freq in counter.items():
        if freq >= min_freq:
            vocab[token] = len(vocab)

    return vocab


def encode_tokens(tokens, vocab):
    ids = []
    unk = vocab["<UNK>"]
    for t in tokens:
        ids.append(vocab.get(t, unk))
    return ids


def pad_or_truncate(ids, max_len, pad_id=0):
    if len(ids) > max_len:
        return ids[:max_len]
    if len(ids) < max_len:
        ids = ids + [pad_id] * (max_len - len(ids))
    return ids

In [ ]:
# Build labels for second task (category prediction)
category_to_id = {
    "beauty": 0,
    "cellphones": 1,
    "sports": 2,
}
id_to_category = {v: k for k, v in category_to_id.items()}

# Build vocabulary from training data only
vocab = build_vocab(train_rows, min_freq=MIN_FREQ)
print("Vocab size:", len(vocab))


def process_rows(rows, vocab, max_len):
    x_data = []
    y_sent = []
    y_cat = []

    for row in rows:
        text = clean_text(row["text"])
        tokens = tokenize(text)
        ids = encode_tokens(tokens, vocab)
        ids = pad_or_truncate(ids, max_len=max_len, pad_id=vocab["<PAD>"])

        x_data.append(ids)
        y_sent.append(rating_to_sentiment(row["rating"]))
        y_cat.append(category_to_id[row["category"]])

    x_data = torch.tensor(x_data, dtype=torch.long)
    y_sent = torch.tensor(y_sent, dtype=torch.long)
    y_cat = torch.tensor(y_cat, dtype=torch.long)

    return x_data, y_sent, y_cat


X_train, y_train_sent, y_train_cat = process_rows(train_rows, vocab, MAX_LEN)
X_val, y_val_sent, y_val_cat = process_rows(val_rows, vocab, MAX_LEN)
X_test, y_test_sent, y_test_cat = process_rows(test_rows, vocab, MAX_LEN)

print("X_train shape:", tuple(X_train.shape))
print("X_val shape:  ", tuple(X_val.shape))
print("X_test shape: ", tuple(X_test.shape))

In [ ]:
# Save processed data for next sections
prep_data = {
    "X_train": X_train,
    "X_val": X_val,
    "X_test": X_test,
    "y_train_sent": y_train_sent,
    "y_val_sent": y_val_sent,
    "y_test_sent": y_test_sent,
    "y_train_cat": y_train_cat,
    "y_val_cat": y_val_cat,
    "y_test_cat": y_test_cat,
    "vocab": vocab,
    "category_to_id": category_to_id,
    "id_to_category": id_to_category,
    "max_len": MAX_LEN,
}

torch.save(prep_data, os.path.join(RESULTS_DIR, "prep_data.pt"))
print("Saved:", os.path.join(RESULTS_DIR, "prep_data.pt"))

# Quick sanity check
print("Example token ids (first 20):", X_train[0][:20].tolist())
print("Example labels (sentiment, category):", y_train_sent[0].item(), y_train_cat[0].item())